# Fine-tuning QLoRA — Professions de foi (Archelec)

**Modèle :** `meta-llama/Llama-3.2-1B-Instruct`  
**GPU cible :** Tesla T4 (fp16, pas de bf16)  
**Objectif :** apprendre au modèle à générer des professions de foi crédibles à partir d'un prompt structuré.

**Pipeline :**
1. Authentification HuggingFace
2. Chargement dataset + métadonnées
3. Construction des prompts
4. Nettoyage OCR
5. Formatage chat template
6. Chargement modèle (QLoRA 4-bit)
7. Entraînement
8. Inférence & comparaison

In [1]:
# 1. Suppression forcée des versions existantes incompatibles
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --no-cache-dir

# 2. Installation stricte de la version CUDA 11.8
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 113.6 MB/s  0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 202.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 115.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 102.4 MB/s  0:00:0600:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 109.9 MB/s  0:00:03a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 113.1 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 113.9 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 95.5 MB/s  0:00:01ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 112.2 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 112.2 MB/s  0:00:01a 0:00:0

In [1]:
import torch

print("Version de PyTorch :", torch.__version__)
print("Version de CUDA associée :", torch.version.cuda)
print("Le GPU est-il détecté ? :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

Version de PyTorch : 2.6.0+cu124
Version de CUDA associée : 12.4
Le GPU est-il détecté ? : True
Nom du GPU : Tesla T4


## 1. Authentification HuggingFace

In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2. Chargement du dataset et des métadonnées

On charge les fichiers texte (professions de foi OCRisées) et le CSV de métadonnées,
puis on les joint par l'identifiant de fichier (`id`).

In [4]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd

# --- Fichiers texte à charger ---
PATTERNS = [
    "data/1981/legislatives/*PF*.txt",
    "data/1988/legislatives/*PF*.txt",
    "data/1993/legislatives/*PF*.txt",
]

# On résout les chemins dans le même ordre que load_dataset
# Note : load_dataset trie les fichiers par glob, on fait pareil avec sorted()
files = []
for p in PATTERNS:
    files.extend(sorted(Path().glob(p)))

dataset = load_dataset("text", data_files=PATTERNS, split="train", sample_by="document")
print(f"Dataset chargé : {len(dataset)} documents, {len(files)} fichiers résolus")

# --- Métadonnées ---
metadata = pd.read_csv("data/archelect_search.csv")
# Colonnes utiles uniquement
COLS_META = ["id", "titulaire-prenom", "titulaire-nom", "titulaire-profession",
             "titulaire-soutien", "contexte-tour", "date", "departement-nom"]
metadata_dict = metadata[COLS_META].set_index("id").to_dict("index")

print(f"Métadonnées : {len(metadata)} entrées")

Generating train split: 12498 examples [00:03, 4013.72 examples/s]


Dataset chargé : 12498 documents, 12498 fichiers résolus
Métadonnées : 12498 entrées


In [5]:
def add_metadata(example, idx):
    """Ajoute l'id (dérivé du nom de fichier) et les métadonnées associées."""
    path = files[idx]
    doc_id = path.stem
    example["id"] = doc_id
    example["annee"] = path.parts[-3]

    # Récupération des métadonnées (avec valeur par défaut si absent)
    meta = metadata_dict.get(doc_id, {})
    example["prenom"]     = meta.get("titulaire-prenom", "non mentionné")
    example["nom"]        = meta.get("titulaire-nom", "non mentionné")
    example["profession"] = meta.get("titulaire-profession", "non mentionné")
    example["soutien"]    = meta.get("titulaire-soutien", "non mentionné")
    example["tour"]       = meta.get("contexte-tour", "non mentionné")
    example["date"]       = meta.get("date", "non mentionné")
    example["departement"]= meta.get("departement-nom", "non mentionné")
    return example

dataset = dataset.map(add_metadata, with_indices=True)
print(dataset)

Map: 100%|██████████| 12498/12498 [00:00<00:00, 19226.87 examples/s]

Dataset({
    features: ['text', 'id', 'annee', 'prenom', 'nom', 'profession', 'soutien', 'tour', 'date', 'departement'],
    num_rows: 12498
})


## 3. Construction des prompts

Le prompt résume les métadonnées du candidat en langage naturel.
Il sera utilisé comme message `user` dans le chat template.

In [6]:
def build_prompt(example):
    """Construit un prompt en langage naturel à partir des métadonnées."""
    parts = ["Rédige une profession de foi"]

    prenom, nom = example["prenom"], example["nom"]
    if prenom != "non mentionné" and nom != "non mentionné":
        parts.append(f"pour {prenom} {nom},")
    else:
        parts.append("pour le candidat,")

    profession = example["profession"]
    if profession != "non mentionné":
        metiers = [m.strip() for m in profession.split(";")]
        if len(metiers) > 1:
            parts.append(f"de professions {', '.join(metiers[:-1])} et {metiers[-1]},")
        else:
            parts.append(f"de profession {metiers[0]},")

    soutien = example["soutien"]
    if soutien != "non mentionné":
        partis = [p.strip() for p in soutien.split(";")]
        if len(partis) > 1:
            parts.append(f"soutenu par les partis {', '.join(partis[:-1])} et {partis[-1]}")
        else:
            parts.append(f"soutenu par le parti {partis[0]}")

    parts.append(f"au tour {example['tour']} des élections législatives de {example['date']}")
    parts.append(f"dans le département : {example['departement']}.")

    example["prompt"] = " ".join(parts)
    return example

dataset = dataset.map(build_prompt)

# Vérification rapide
print(dataset[0]["prompt"])

Map:   0%|          | 0/12498 [00:00<?, ? examples/s]

Map: 100%|██████████| 12498/12498 [00:01<00:00, 6626.37 examples/s]

Rédige une profession de foi pour Micheline Antonucci, de profession assistance sociale, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Ain.


## 4. Nettoyage OCR

Les textes sont issus d'OCR et contiennent des artefacts courants :
mots coupés par un tiret en fin de ligne, filigranes CEVIPOF, mentions légales, etc.

**Attention :** on ne colle PAS les tirets intra-ligne (ex. `Bourg-en-Bresse`)
car cela détruirait les noms composés. On traite uniquement les coupures
**de fin de ligne** (tiret suivi d'un saut de ligne).

In [7]:
import re

# Précompilation pour performances (appliqué sur 12k documents)
_RE_CUT_EOL    = re.compile(r'([A-Za-zÀ-ÿ]+)-\s*\n\s*([A-Za-zÀ-ÿ]+)')  # coupure fin de ligne uniquement
_RE_WATERMARK  = re.compile(r'Sciences Po / fonds CEVIPOF|[☐☒@¥]')
_RE_VU_CAND    = re.compile(r'vu\s*[,:\-]?\s*(le|la|les)\s+candidat[e]?[s]?\s*[:.]?', re.IGNORECASE)
_RE_DROP_LINE  = re.compile(
    r'^.*('
    r'imp\.?\s|imprimerie|imprimeurs|'
    r'r\.?c\.?\s|'
    r'\b\d{5}\b|'
    r'\b\d{1,2}([\s.\-]?\d{2}){3}\b'
    r').*$',
    re.IGNORECASE | re.MULTILINE
)
_RE_MULTILINE  = re.compile(r'\n{3,}')
_RE_MULTSPACE  = re.compile(r' {2,}')
_RE_ENUM       = re.compile(r'[:\n]\s*[•*>.o·]\s*', re.MULTILINE)

def clean_ocr(example):
    text = example["text"]
    text = _RE_CUT_EOL.sub(r'\1\2', text)     # recolle mots coupés en fin de ligne
    text = _RE_WATERMARK.sub("", text)          # filigranes
    text = _RE_VU_CAND.sub("", text)            # mention légale
    text = _RE_DROP_LINE.sub("", text)          # lignes techniques
    text = _RE_MULTILINE.sub('\n\n', text)      # sauts de ligne excessifs
    text = _RE_MULTSPACE.sub(' ', text)         # espaces multiples
    text = _RE_ENUM.sub("- ", text)             # normalisation listes
    example["text"] = text.strip()
    return example

dataset = dataset.map(clean_ocr, num_proc=4)
print("Nettoyage OCR terminé.")

Map (num_proc=4): 100%|██████████| 12498/12498 [00:07<00:00, 1772.84 examples/s]


Nettoyage OCR terminé.


## 5. Formatage avec le chat template de Llama-3.2

On construit les paires `(user, assistant)` et on applique le template du tokenizer.
La colonne `formatted_text` sera celle passée au trainer.

In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# On charge le tokenizer ici uniquement pour apply_chat_template
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right pour l'entraînement (causal LM)

def apply_chat_template(example):
    """Formate le couple (prompt, texte) avec le template de conversation du modèle."""
    messages = [
        {"role": "user",      "content": example["prompt"]},
        {"role": "assistant", "content": example["text"]},
    ]
    example["formatted_text"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return example

dataset = dataset.map(apply_chat_template)

# Vérification
print(dataset[0]["formatted_text"][:500])
print("...")

Map: 100%|██████████| 12498/12498 [00:03<00:00, 3714.57 examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 08 Apr 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Rédige une profession de foi pour Micheline Antonucci, de profession assistance sociale, soutenu par le parti Parti socialiste unifié au tour 1 des élections législatives de 1981-06-14 dans le département : Ain.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

ELECTIONS LEGISLATIVES - 14 JUIN 1981 AIN 1e CIRCONSCRIP
...


In [9]:
# Split train / test (90% / 10%)
splits = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = splits["train"]
test_dataset  = splits["test"]

print(f"Train : {len(train_dataset)} | Test : {len(test_dataset)}")

Train : 11248 | Test : 1250


## 6. Chargement du modèle (QLoRA 4-bit)

**Points critiques pour la Tesla T4 :**
- La T4 **ne supporte pas `bf16`** (bfloat16). Il faut utiliser `fp16` (float16).
- On spécifie `torch_dtype=torch.float16` dès le chargement pour que Llama-3.2 
  charge directement ses couches non-quantifiées (lm_head, embed_tokens) en float16.
- Pas besoin de boucle de cast manuel après coup.

## 7. Entraînement

**Paramètres importants pour la T4 :**
- `fp16=True`, `bf16=False` — indispensable sur T4
- `optim="paged_adamw_32bit"` — optimiseur paginé du papier QLoRA, réduit la VRAM
- `max_seq_length=1024` — les professions de foi font souvent ~400-800 tokens,
  1024 est un bon compromis vitesse/couverture (2048 doublerait l'usage VRAM)
- `packing=True` — emballe plusieurs courtes séquences dans un même batch : accélère notablement l'entraînement

In [11]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

# --- 1. Paramètres globaux ---
max_seq_length = 1024
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# --- 2. Chargement du modèle optimisé par Unsloth ---
# dtype=None permet à Unsloth de détecter automatiquement la T4 et de forcer le float16.
# load_in_4bit=False respecte votre choix de ne pas utiliser la quantification pour le moment.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=False, 
)

# --- 3. Configuration de l'adaptateur LoRA ---
# Unsloth réécrit les calculs sous le capot pour accélérer cette étape.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    # lora_dropout DOIT être à 0 pour bénéficier de l'accélération mathématique d'Unsloth
    lora_dropout=0, 
    bias="none",
    # Utilisation du checkpointing propriétaire d'Unsloth (réduit la VRAM de 30%)
    use_gradient_checkpointing="unsloth", 
    random_state=42,
)

print("Modèle chargé et adaptateur LoRA configuré avec succès via Unsloth.")

# --- 4. Nettoyage strict des datasets ---
# Nécessaire pour éviter l'erreur KeyError: 'completion' du SFTTrainer
colonnes_a_supprimer = [col for col in train_dataset.column_names if col != "formatted_text"]
train_dataset_clean = train_dataset.remove_columns(colonnes_a_supprimer)
test_dataset_clean = test_dataset.remove_columns(colonnes_a_supprimer)

# --- 5. Configuration de l'entraînement ---
training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    
    # Maintien du batch size à 2 pour garantir la stabilité de la VRAM
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    
    # Précision stricte pour la Tesla T4
    fp16=True,
    bf16=False,
    
    # adamw_8bit consomme très peu de VRAM, idéal pour compenser l'absence de quantification
    optim="adamw_8bit",
    weight_decay=0.01,
    
    dataset_text_field="formatted_text",
    max_length=max_seq_length,
    packing=False,
    
    # Suivi et validation
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    report_to="none",
)

# --- 6. Initialisation et lancement du Trainer ---
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_clean,
    eval_dataset=test_dataset_clean,
    processing_class=tokenizer,
    args=training_args,
)

# Lance l'entraînement optimisé
trainer.train()

# --- 7. Sauvegarde ---
# Méthode spécifique à Unsloth pour sauvegarder uniquement l'adaptateur
model.save_pretrained("Llama-1B-Archelec-Unsloth-LoRA")
tokenizer.save_pretrained("Llama-1B-Archelec-Unsloth-LoRA")
print("Entraînement terminé. Modèle sauvegardé.")

/tmp/ipykernel_8808/3963642795.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed editing tqdm to replace Inductor Compilation:


Exception: cannot import name 'pw_cast_for_opmath_non_tensor_args' from 'torch._decomp.decompositions' (/opt/python/lib/python3.13/site-packages/torch/_decomp/decompositions.py)

## 8. Inférence & comparaison base vs fine-tuné

On charge les poids LoRA sur le modèle de base et on génère une profession de foi test.
On compare avec le modèle de base (sans LoRA) pour mesurer l'effet du fine-tuning.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

ADAPTER_PATH = "Llama-1B-Archelec-LoRA"

# --- Tokenizer (padding LEFT pour la génération) ---
tokenizer_inf = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer_inf.padding_side = "left"

# --- Modèle de base en 4-bit ---
bnb_config_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config_inf,
    torch_dtype=torch.float16,
)

# --- Chargement de l'adaptateur LoRA ---
model_ft = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

print("Modèle fine-tuné chargé.")

In [ ]:
def generate(model, tokenizer, prompt_text, max_new_tokens=400):
    """Génère une réponse à partir d'un prompt utilisateur."""
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    # Isole la réponse de l'assistant
    return decoded.split("assistant\n")[-1].strip() if "assistant\n" in decoded else decoded


# --- Prompt de test ---
TEST_PROMPT = (
    "Rédige une profession de foi pour Jean Dupont, de profession professeur, "
    "soutenu par le parti Parti communiste français "
    "au tour 1 des élections législatives de 1981-06-14 "
    "dans le département : Paris."
)

print("=" * 60)
print("MODÈLE FINE-TUNÉ (Archelec-LoRA)")
print("=" * 60)
print(generate(model_ft, tokenizer_inf, TEST_PROMPT))

print("\n" + "=" * 60)
print("MODÈLE DE BASE (Llama-3.2-1B-Instruct sans fine-tuning)")
print("=" * 60)
with model_ft.disable_adapter():
    print(generate(model_ft, tokenizer_inf, TEST_PROMPT))